# 06. Self-Attention: как токены учитывают друг друга

Мы уже подготовили последовательность векторов: token embeddings плюс positional encoding. Теперь разберём центральный механизм Transformer — **Self-Attention**.

Self-Attention позволяет каждому токену вычислять, насколько важна информация от каждого другого токена в той же последовательности.

В этом блокноте сначала вручную построим механизм из отдельных матричных операций, а затем соберём его в класс PyTorch.

## 1. Интуиция: что делает Self-Attention?

Представим предложение `The movie was not good`. Когда модель обрабатывает токен `good`, ей может быть полезно учитывать токен `not`. А когда она обрабатывает `movie`, ей может быть полезно учитывать другие слова, описывающие фильм.

Self-Attention позволяет каждому токену собрать информацию из других позиций предложения. При этом модель не получает готовое правило вроде «not меняет good»: она учится вычислять связи на основе обучаемых параметров.

Для этого из каждого входного вектора создаются три новых вектора:

- **Query (Q, запрос)** — условно, какую информацию ищет текущий токен;
- **Key (K, ключ)** — условно, по каким признакам другие токены могут быть ему полезны;
- **Value (V, значение)** — информация, которую токен передаёт другим позициям.

Это полезная аналогия, но математически Q, K и V — просто три разных обучаемых линейных преобразования входных векторов.

## 2. Подготовим входные векторы

Пока не подключаем реальный датасет. Создадим маленький искусственный пример: одно предложение из трёх токенов, каждый токен представлен четырьмя числами.

В реальной модели на это место поступит сумма token embedding и positional encoding.

In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

# Batch из одного предложения, три токена, размерность каждого вектора — 4.
X = torch.tensor([[
    [0.2, 0.1, 0.7, 0.3],
    [0.8, 0.4, 0.1, 0.5],
    [0.3, 0.9, 0.2, 0.6]
]])

batch_size, sequence_length, d_model = X.shape

print("X:\n", X)
print("Форма X:", X.shape)
print("Batch size:", batch_size)
print("Sequence length:", sequence_length)
print("d_model:", d_model)

Форма `X` — `(1, 3, 4)`:

- `1` — одно предложение в batch;
- `3` — три токена;
- `4` — четыре числа в каждом векторе.

Важное соглашение: дальше будем использовать формат `(batch, sequence_length, features)`, то есть `(B, T, C)`.

## 3. Получаем Query, Key и Value

Создадим три линейных слоя: `q_proj`, `k_proj` и `v_proj`. Каждый слой преобразует входной вектор размерности 4 в новый вектор размерности 4.

Линейный слой имеет обучаемые веса. В упрощённой математической записи это можно представить так:

`Q = X W_Q`, `K = X W_K`, `V = X W_V`.

В PyTorch `nn.Linear` хранит матрицу весов в форме `(out_features, in_features)` и самостоятельно выполняет нужное преобразование. Поэтому не удивляйся, что форма `layer.weight` отличается от формы матрицы в записи `X @ W`.

In [ ]:
d_k = d_model

q_proj = nn.Linear(d_model, d_k, bias=False)
k_proj = nn.Linear(d_model, d_k, bias=False)
v_proj = nn.Linear(d_model, d_k, bias=False)

Q = q_proj(X)
K = k_proj(X)
V = v_proj(X)

print("Форма Q:", Q.shape)
print("Форма K:", K.shape)
print("Форма V:", V.shape)
print("\nQ для первого предложения:\n", Q[0])
print("\nK для первого предложения:\n", K[0])
print("\nV для первого предложения:\n", V[0])

### Что важно понять на этом этапе?

Q, K и V получены из одних и тех же входных векторов `X`, но с помощью разных наборов весов.

Именно поэтому это **self**-attention: запросы, ключи и значения строятся из одной последовательности. В cross-attention, который встречается, например, в encoder-decoder моделях, Q и K/V могут поступать из разных последовательностей.

Пока веса линейных слоёв случайны. Поэтому полученные числа ещё не отражают осмысленные языковые закономерности. Они станут полезными в процессе обучения.

## 4. Считаем совместимость Query и Key

Теперь модель сравнивает каждый Query со всеми Key. Для этого используется скалярное произведение.

Для каждого предложения вычисляем:

`scores = Q @ K.transpose(-2, -1)`

Почему транспонируем K? Нам нужно умножить матрицу формы `(T, d_k)` на матрицу формы `(d_k, T)`. Получится матрица `(T, T)`.

Строка этой матрицы соответствует токену, который задаёт запрос. Столбец — токену, чей Key сравнивается с этим запросом.

In [ ]:
scores = Q @ K.transpose(-2, -1)

print("Форма Q:", Q.shape)
print("Форма K после transpose:", K.transpose(-2, -1).shape)
print("Матрица scores:\n", scores)
print("Форма scores:", scores.shape)

У нас получилась матрица `(1, 3, 3)`: для каждого из трёх токенов вычислена совместимость с тремя Key.

Элемент `scores[0, 1, 2]` — насколько Query второго токена согласуется с Key третьего токена. Индексация начинается с нуля.

Большое значение означает высокую совместимость в текущем пространстве проекций. Но это ещё не вероятность и не окончательный вес внимания.

## 5. Зачем делить на корень из d_k?

Перед softmax оценки делят на `sqrt(d_k)`:

`scaled_scores = scores / sqrt(d_k)`

Это называется **scaled dot-product attention**.

Почему это нужно? Когда размерность векторов велика, скалярные произведения могут иметь большие по модулю значения. Тогда softmax может выдавать очень резкое распределение: почти весь вес приходится на один элемент. В таких условиях градиентам может быть сложнее эффективно обучать параметры.

Деление на `sqrt(d_k)` помогает удерживать масштабы оценок в более удобном диапазоне.

In [ ]:
scaled_scores = scores / math.sqrt(d_k)

print("d_k:", d_k)
print("sqrt(d_k):", math.sqrt(d_k))
print("Оценки до масштабирования:\n", scores)
print("\nОценки после масштабирования:\n", scaled_scores)

## 6. Softmax: превращаем оценки во веса внимания

Теперь применим softmax к каждой строке матрицы. Получим веса, которые складываются в единицу по каждой строке.

`attention_weights = softmax(scaled_scores, dim=-1)`

Параметр `dim=-1` означает: применить softmax вдоль последней оси — по столбцам каждой строки.

Таким образом, для каждого токена веса показывают, как он распределяет своё внимание между всеми позициями последовательности.

In [ ]:
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("Веса внимания:\n", attention_weights)
print("\nСумма весов для каждого токена:", attention_weights.sum(dim=-1))

Каждая строка весов должна суммироваться примерно в 1. Это можно понимать как распределение долей внимания данного токена по позициям.

Например, если строка выглядит как `[0.1, 0.7, 0.2]`, то второй токен получил наибольший вес при формировании нового представления текущего токена.

Важно: веса внимания не являются доказательством того, что модель «понимает» соответствующее слово именно так, как человек. Это коэффициенты вычисления нового представления.

## 7. Получаем результат: взвешенная сумма Value

Мы получили веса внимания. Теперь используем их, чтобы объединить векторы V:

`output = attention_weights @ V`

Если для токена веса равны `[0.1, 0.7, 0.2]`, его новое представление будет суммой:

`0.1 * V[0] + 0.7 * V[1] + 0.2 * V[2]`.

Иными словами, каждый токен получает новую комбинацию информации от всех токенов предложения.

In [ ]:
attention_output = attention_weights @ V

print("Форма attention output:", attention_output.shape)
print("Новые представления токенов:\n", attention_output)

### Проверим вычисление для одного токена вручную

Возьмём первый токен и умножим каждый Value на соответствующий вес внимания, затем сложим результаты. Сравним с первой строкой `attention_output`.

In [ ]:
first_token_weights = attention_weights[0, 0]  # веса первого токена
first_token_values = V[0]                     # все Value первого предложения

manual_output = (first_token_weights.unsqueeze(-1) * first_token_values).sum(dim=0)

print("Веса первого токена:", first_token_weights)
print("Результат вручную:", manual_output)
print("Результат через матричное умножение:", attention_output[0, 0])
print("Совпадают:", torch.allclose(manual_output, attention_output[0, 0]))

## 8. Собираем Self-Attention в отдельный модуль

Выше мы выполнили каждый шаг отдельно. Теперь объединим их в класс `SelfAttention`, чтобы позднее подключить его к Transformer Block.

`nn.Module` — базовый класс PyTorch для компонентов нейросети. Когда мы создаём слои внутри такого класса, PyTorch может отслеживать их обучаемые параметры.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x имеет форму (batch, sequence_length, d_model)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Сравниваем каждый Query со всеми Key.
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.d_model)

        # Нормируем оценки по позициям Key.
        weights = torch.softmax(scores, dim=-1)

        # Собираем информацию из Value.
        context = weights @ v

        # Дополнительное обучаемое преобразование результата.
        output = self.out_proj(context)
        return output, weights


self_attention = SelfAttention(d_model)
module_output, module_weights = self_attention(X)

print("Форма выхода:", module_output.shape)
print("Форма весов внимания:", module_weights.shape)
print("Количество параметров модуля:", sum(p.numel() for p in self_attention.parameters()))

### Что мы пока не реализовали?

Это базовая версия self-attention для короткой последовательности. В ней пока нет:

- **Attention mask для padding** — она нужна, чтобы модель не учитывала позиции `<PAD>`;
- **Causal mask** — она запрещает смотреть на будущие токены и используется в авторегрессионных GPT-подобных моделях;
- **Multi-Head Attention** — несколько механизмов внимания, работающих параллельно;
- **Dropout и residual connection** — важные элементы полноценного Transformer Block.

Для нашей первой задачи — классификации тональности — мы строим encoder-style модель: токен может учитывать слова и слева, и справа. Поэтому causal mask нам не нужна. Маску padding добавим отдельно.

## 9. Самостоятельные упражнения

1. Измени `sequence_length`: добавь четвёртый токен во вход `X`. Какая форма получится у матрицы `scores` и почему?
2. Измени `d_model` с 4 на 8. Не забудь заново создать входные данные и проекции. Как изменится форма Q, K и V?
3. Проверь, что суммы строк `attention_weights` равны 1 (с учётом погрешности вычислений).
4. Вычисли вручную взвешенную сумму Value для второго токена и сравни её с `attention_output[0, 1]`.
5. Объясни своими словами, что такое Query, Key и Value.
6. Почему в Self-Attention матрица `scores` имеет форму `(batch, sequence_length, sequence_length)`?

Не нужно пытаться запомнить все формулы сразу. Важно понимать последовательность операций: проекции Q/K/V → скалярные произведения → масштабирование → softmax → взвешенная сумма V.

## Что мы узнали

Мы реализовали одно-головное Self-Attention:

1. Из входных векторов получили Q, K и V.
2. Сравнили Query и Key с помощью скалярных произведений.
3. Разделили оценки на `sqrt(d_k)`.
4. Применили softmax и получили веса внимания.
5. Использовали веса для взвешенной суммы Value.

Следующий шаг — **маски внимания и Multi-Head Attention**. Разберём, как не учитывать padding и как запускать несколько голов внимания параллельно.